# 12. Ames Mutagenicity 예측 모델 추가

## 이번 노트북에서 할 것
- Ames test(변이원성) 라벨이 있는 공개 데이터셋 확보
- Tox21 baseline과 동일한 방식(ECFP + RandomForest)으로 학습
- alkyl_halide, nitro_group 등 구조적 위험 신호와 실제 변이원성 예측이
  더 직접적으로 연결되는지 확인 (Tox21 12개 assay와는 다른 관점의 검증)
- iterative_fix_loop의 재평가 단계에 Ames 예측 결과 추가 통합

## 간략한 정리 (11까지)
- Held-out 평가 완료: 단일문제 50개 표본(58% 완전해결+36% 부분개선=94% 실질개선),
  다중문제 7개 전수 success
- 규칙기반 vs LLM기반 candidate 비교(10개): 40%에서 다른 선택, 다할로겐 사슬
  분자에서 LLM이 4회 연속 일관되게 fluorine 선택
- 심화 발견: FilterCatalog(구조기반 경고)와 Tox21 12개 assay(실험기반)는
  서로 다른 독성 차원을 측정함을 확인 (alkyl_halide 개선 시 12개 assay 중
  7개 개선/5개 악화로 혼재, 이는 alkylation 관련 assay 자체가 Tox21에 없기 때문)
- 이 발견 때문에, 구조적 위험(알킬화 반응성)과 더 직접 연결되는 Ames test
  모델을 추가하기로 결정

## 다음에 해야 할 것 (오늘 끝나면)
- Ames 모델까지 완성되면, 3중 검증 체계(구조규칙+Tox21+Ames) 완성
- 제안서 4번(평가) 섹션에 이 확장된 검증 체계 반영
- 제안서 전체 초안 마무리 및 다듬기 (마감 8/7)

In [1]:
# 셀 1
!pip install rdkit PyTDC -q

In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
from rdkit import Chem
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()
print("기존 도구 로드 확인 완료")

[07:03:45] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7831개, 파싱 실패(제외): 0개


[07:03:48] WARNING: not removing hydrogen atom without neighbors


기존 도구 로드 확인 완료


In [5]:
from tdc.single_pred import Tox

ames_data = Tox(name='AMES')
split = ames_data.get_split()

print("Train:", split['train'].shape)
print("Valid:", split['valid'].shape)
print("Test:", split['test'].shape)
print(split['train'].head())

Downloading...
100%|██████████| 344k/344k [00:00<00:00, 517kiB/s]
Loading...
Done!


Train: (5094, 3)
Valid: (728, 3)
Test: (1456, 3)
  Drug_ID                                               Drug  Y
0  Drug 1       O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2  1
1  Drug 2  O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...  0
2  Drug 3                          [N-]=[N+]=CC(=O)NCC(=O)NN  1
3  Drug 4                          [N-]=[N+]=C1C=NC(=O)NC1=O  1
4  Drug 6          CCCCN(CC(O)C1=CC(=[N+]=[N-])C(=O)C=C1)N=O  1


In [6]:
import numpy as np
from rdkit.Chem import rdFingerprintGenerator

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return _generator.GetFingerprintAsNumPy(mol)

def prepare_ames_split(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    n_total, n_valid = len(df), df['mol_valid'].sum()
    print(f"  전체: {n_total}개, 파싱 성공: {n_valid}개, 실패(제외): {n_total-n_valid}개")
    df_clean = df[df['mol_valid']].reset_index(drop=True)

    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    smiles_arr = df_clean['Drug'].values
    return X, y, smiles_arr

print("Train 처리 중:")
X_train_ames, y_train_ames, smiles_train_ames = prepare_ames_split(split['train'])
print("Valid 처리 중:")
X_valid_ames, y_valid_ames, smiles_valid_ames = prepare_ames_split(split['valid'])
print("Test 처리 중:")
X_test_ames, y_test_ames, smiles_test_ames = prepare_ames_split(split['test'])

print("\nTrain shape:", X_train_ames.shape)
print("양성 비율:", y_train_ames.mean())

Train 처리 중:
  전체: 5094개, 파싱 성공: 5094개, 실패(제외): 0개
Valid 처리 중:
  전체: 728개, 파싱 성공: 728개, 실패(제외): 0개
Test 처리 중:
  전체: 1456개, 파싱 성공: 1456개, 실패(제외): 0개

Train shape: (5094, 2048)
양성 비율: 0.5416175893207695


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

ames_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
ames_clf.fit(X_train_ames, y_train_ames)

valid_probs = ames_clf.predict_proba(X_valid_ames)[:, 1]
valid_auc = roc_auc_score(y_valid_ames, valid_probs)
print(f"Valid AUROC: {valid_auc:.3f}")

test_probs = ames_clf.predict_proba(X_test_ames)[:, 1]
test_auc = roc_auc_score(y_test_ames, test_probs)
print(f"Test AUROC: {test_auc:.3f}")

Valid AUROC: 0.892
Test AUROC: 0.908


In [8]:
import joblib
import json
from datetime import datetime

joblib.dump(ames_clf, 'models/tox21_classifier/ames_rf.pkl')

ames_metadata = {
    "model_type": "RandomForestClassifier",
    "featurizer": "ECFP (radius=2, n_bits=2048)",
    "dataset": "Hansen et al. 2009 Ames Mutagenicity Benchmark (via TDC)",
    "valid_auc": float(valid_auc),
    "test_auc": float(test_auc),
    "created_at": datetime.now().isoformat(),
}
with open('models/tox21_classifier/ames_rf_meta.json', 'w') as f:
    json.dump(ames_metadata, f, indent=2)

print("저장 완료")

저장 완료


In [9]:
def predict_ames(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol).reshape(1, -1)
    return ames_clf.predict_proba(fp)[0][1]

original = "CCOP(=S)(OCC)OC(Cl)C(Cl)(Cl)Cl"
llm_result_final = "CCOP(=S)(OCC)OC(F)C(F)(F)F"

print(f"원본 Ames 변이원성 예측: {predict_ames(original):.3f}")
print(f"치환후 Ames 변이원성 예측: {predict_ames(llm_result_final):.3f}")

원본 Ames 변이원성 예측: 0.340
치환후 Ames 변이원성 예측: 0.160


In [10]:
!git add src/tools/ models/tox21_classifier/ames_rf_meta.json
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   models/tox21_classifier/ames_rf_meta.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/ames.tab
	laidd-2026/



In [11]:
!git commit -m "Add Ames mutagenicity model (Hansen 2009 via TDC, AUROC 0.908); confirms alkyl_halide->fluorine substitution reduces predicted mutagenicity (0.340->0.160), resolving earlier ambiguous Tox21 12-assay result by using mechanistically relevant endpoint"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main cce7a80] Add Ames mutagenicity model (Hansen 2009 via TDC, AUROC 0.908); confirms alkyl_halide->fluorine substitution reduces predicted mutagenicity (0.340->0.160), resolving earlier ambiguous Tox21 12-assay result by using mechanistically relevant endpoint
 1 file changed, 8 insertions(+)
 create mode 100644 models/tox21_classifier/ames_rf_meta.json
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 821 bytes | 821.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Dec32th/laidd-2026.git
   7d95fa5..cce7a80  main -> main


In [12]:
def predict_ames(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = _generator.GetFingerprintAsNumPy(mol).reshape(1, -1)
    return ames_clf.predict_proba(fp)[0][1]

def predict_tox21_avg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = _generator.GetFingerprintAsNumPy(mol).reshape(1, -1)
    scores = [classifiers[t].predict_proba(fp)[0][1] for t in task_cols]
    return np.mean(scores)

# alkyl_halide, nitro_group 문제를 가진 분자들을 test set에서 여러 개 수집
target_rules = ["alkyl_halide", "nitro_group"]
verification_cases = []

for s in data['smiles_test']:
    problems = detect_toxicophores(s)
    known = [p for p in problems if p['rule_name'] in target_rules]
    if not known:
        continue
    rule = known[0]['rule_name']
    fixed = propose_fix(s, rule, candidate_idx=0)  # 일단 규칙기반(0번 후보)으로 비교
    if fixed is None or not fixed['is_valid']:
        continue
    verification_cases.append({"original": s, "fixed": fixed['new_smiles'], "rule": rule})
    if len(verification_cases) >= 15:
        break

print(f"검증 대상 확보: {len(verification_cases)}개")
for c in verification_cases:
    print(f"  [{c['rule']}] {c['original'][:40]} -> {c['fixed'][:40]}")

검증 대상 확보: 15개
  [alkyl_halide] CC(Cl)(Cl)Cl -> CC(O)(Cl)Cl
  [alkyl_halide] ClC(Cl)(Cl)C(Cl)(Cl)Cl -> OC(Cl)(Cl)C(Cl)(Cl)Cl
  [nitro_group] Cc1cccc(C(=O)O)c1[N+](=O)[O-] -> Cc1cccc(C(=O)O)c1N
  [alkyl_halide] O=C1CCCCC1Cl -> O=C1CCCCC1O
  [nitro_group] COC(=O)C1=C(C)NC(C)=C(C(=O)OCCc2ccc(N3CC -> COC(=O)C1=C(C)NC(C)=C(C(=O)OCCc2ccc(N3CC
  [alkyl_halide] O=C(Cl)C(Cl)Cl -> O=C(Cl)C(O)Cl
  [alkyl_halide] CC(Cl)(Cl)C(=O)O -> CC(O)(Cl)C(=O)O
  [nitro_group] O=[N+]([O-])c1ccc(Oc2ccc(C(F)(F)F)cc2[N+ -> Nc1ccc(Oc2ccc(C(F)(F)F)cc2[N+](=O)[O-])c
  [alkyl_halide] O=C(CCCl)NCc1ccccc1 -> O=C(CCO)NCc1ccccc1
  [alkyl_halide] CC(=O)Oc1ccc([N+](=O)[O-])cc1CCl -> CC(=O)Oc1ccc([N+](=O)[O-])cc1CO
  [alkyl_halide] Cc1cc(Br)ccc1NC(=O)CCl -> Cc1cc(O)ccc1NC(=O)CCl
  [nitro_group] O=[N+]([O-])c1ccccc1CCO -> Nc1ccccc1CCO
  [alkyl_halide] CN(C)CCCCl -> CN(C)CCCO
  [alkyl_halide] NS(=O)(=O)c1cc2c(cc1Cl)NC(C(Cl)Cl)NS2(=O -> NS(=O)(=O)c1cc2c(cc1O)NC(C(Cl)Cl)NS2(=O)
  [nitro_group] O=[N+]([O-])C([N+](=O)[O-])([N+](=O

In [17]:
from sklearn.ensemble import RandomForestClassifier

X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']

classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf

print("Tox21 baseline 재학습 완료, task 개수:", len(classifiers))

Tox21 baseline 재학습 완료, task 개수: 12


In [18]:
task_cols = data['task_cols']
print(task_cols)

['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [19]:
import numpy as np
from rdkit.Chem import rdFingerprintGenerator

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
results_compare = []

for c in verification_cases:
    orig_tox21 = predict_tox21_avg(c['original'])
    fixed_tox21 = predict_tox21_avg(c['fixed'])
    orig_ames = predict_ames(c['original'])
    fixed_ames = predict_ames(c['fixed'])

    if None in (orig_tox21, fixed_tox21, orig_ames, fixed_ames):
        continue

    results_compare.append({
        "rule": c['rule'],
        "tox21_change": fixed_tox21 - orig_tox21,
        "ames_change": fixed_ames - orig_ames,
    })

import numpy as np
tox21_changes = [r['tox21_change'] for r in results_compare]
ames_changes = [r['ames_change'] for r in results_compare]

print(f"검증된 쌍: {len(results_compare)}개\n")
print(f"Tox21 평균 변화: {np.mean(tox21_changes):+.4f} (표준편차 {np.std(tox21_changes):.4f})")
print(f"  개선(감소): {sum(1 for x in tox21_changes if x < 0)}개")
print(f"  악화(증가): {sum(1 for x in tox21_changes if x > 0)}개")
print()
print(f"Ames 평균 변화: {np.mean(ames_changes):+.4f} (표준편차 {np.std(ames_changes):.4f})")
print(f"  개선(감소): {sum(1 for x in ames_changes if x < 0)}개")
print(f"  악화(증가): {sum(1 for x in ames_changes if x > 0)}개")

검증된 쌍: 15개

Tox21 평균 변화: -0.0032 (표준편차 0.0288)
  개선(감소): 8개
  악화(증가): 6개

Ames 평균 변화: -0.1313 (표준편차 0.2345)
  개선(감소): 9개
  악화(증가): 6개


In [20]:
from scipy import stats

t_stat, p_value = stats.ttest_1samp(ames_changes, 0)
print(f"Ames 변화가 0과 다른지 검정: t={t_stat:.3f}, p={p_value:.4f}")

t_stat2, p_value2 = stats.ttest_1samp(tox21_changes, 0)
print(f"Tox21 변화가 0과 다른지 검정: t={t_stat2:.3f}, p={p_value2:.4f}")

Ames 변화가 0과 다른지 검정: t=-2.095, p=0.0548
Tox21 변화가 0과 다른지 검정: t=-0.422, p=0.6797


In [24]:
# alkyl_halide, nitro_group 외 규칙도 포함해서 표본 확대 시도
target_rules_expanded = ["alkyl_halide", "nitro_group", "aniline", "acyl_halide", "aldehyde"]

verification_cases2 = []
for s in data['smiles_test']:
    problems = detect_toxicophores(s)
    known = [p for p in problems if p['rule_name'] in target_rules_expanded]
    if not known:
        continue
    rule = known[0]['rule_name']
    fixed = propose_fix(s, rule, candidate_idx=0)
    if fixed is None or not fixed['is_valid']:
        continue
    verification_cases2.append({"original": s, "fixed": fixed['new_smiles'], "rule": rule})

print(f"확장된 검증 대상: {len(verification_cases2)}개")

[07:43:18] Incomplete atom labelling, cannot make bond
[07:43:19] Incomplete atom labelling, cannot make bond


확장된 검증 대상: 167개


In [22]:
results_compare2 = []

for c in verification_cases2:
    orig_tox21 = predict_tox21_avg(c['original'])
    fixed_tox21 = predict_tox21_avg(c['fixed'])
    orig_ames = predict_ames(c['original'])
    fixed_ames = predict_ames(c['fixed'])

    if None in (orig_tox21, fixed_tox21, orig_ames, fixed_ames):
        continue

    results_compare2.append({
        "rule": c['rule'],
        "tox21_change": fixed_tox21 - orig_tox21,
        "ames_change": fixed_ames - orig_ames,
    })

tox21_changes2 = [r['tox21_change'] for r in results_compare2]
ames_changes2 = [r['ames_change'] for r in results_compare2]

print(f"검증된 쌍: {len(results_compare2)}개\n")
print(f"Tox21 평균 변화: {np.mean(tox21_changes2):+.4f} (표준편차 {np.std(tox21_changes2):.4f})")
print(f"  개선: {sum(1 for x in tox21_changes2 if x < 0)}개, 악화: {sum(1 for x in tox21_changes2 if x > 0)}개")
print()
print(f"Ames 평균 변화: {np.mean(ames_changes2):+.4f} (표준편차 {np.std(ames_changes2):.4f})")
print(f"  개선: {sum(1 for x in ames_changes2 if x < 0)}개, 악화: {sum(1 for x in ames_changes2 if x > 0)}개")

t1, p1 = stats.ttest_1samp(tox21_changes2, 0)
t2, p2 = stats.ttest_1samp(ames_changes2, 0)
print(f"\nTox21 검정: t={t1:.3f}, p={p1:.5f}")
print(f"Ames 검정: t={t2:.3f}, p={p2:.5f}")

검증된 쌍: 167개

Tox21 평균 변화: -0.0136 (표준편차 0.0421)
  개선: 110개, 악화: 56개

Ames 평균 변화: -0.1448 (표준편차 0.2369)
  개선: 128개, 악화: 38개

Tox21 검정: t=-4.163, p=0.00005
Ames 검정: t=-7.873, p=0.00000


In [26]:
import warnings

problem_cases = []
for c in verification_cases2:
    core_mol = Chem.MolFromSmiles(c['fixed'])
    if core_mol is None:
        problem_cases.append(c)

print(f"파싱 실패 사례: {len(problem_cases)}개")
for c in problem_cases:
    print(c)

파싱 실패 사례: 0개


In [27]:
!git add -A
!git status

hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> laidd-2026
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached laidd-2026
hint: 
hint: See "git help submodule" for more information.
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/ames.tab
	new file:   laidd-2026



In [28]:
!pwd
!ls

/content/laidd-2026
data  docs  laidd-2026	models	notebooks  outputs  README.md  references  src


In [29]:
!git restore --staged laidd-2026
!git restore --staged data/ames.tab

In [30]:
!echo "data/ames.tab" >> .gitignore
!echo "data/*.tab" >> .gitignore

In [31]:
!rm -rf laidd-2026
!ls

data  docs  models  notebooks  outputs	README.md  references  src


In [32]:
!git add .gitignore
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   .gitignore



In [33]:
!git commit -m "Update .gitignore to exclude TDC-downloaded data files"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 1b60bd3] Update .gitignore to exclude TDC-downloaded data files
 1 file changed, 2 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 352 bytes | 352.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Dec32th/laidd-2026.git
   cce7a80..1b60bd3  main -> main


In [34]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [35]:
!git log --oneline -5
!ls models/tox21_classifier/

1b60bd3 (HEAD -> main) Update .gitignore to exclude TDC-downloaded data files
cce7a80 Add Ames mutagenicity model (Hansen 2009 via TDC, AUROC 0.908); confirms alkyl_halide->fluorine substitution reduces predicted mutagenicity (0.340->0.160), resolving earlier ambiguous Tox21 12-assay result by using mechanistically relevant endpoint
7d95fa5 (origin/main, origin/HEAD) Remove invalid gitlink entry
6952689 Complete held-out evaluation (94% show measurable improvement) + deep-dive analysis: structural alerts (FilterCatalog) and Tox21 assays measure different toxicity dimensions, confirmed via full 12-assay comparison
36d923f Final verification: Case B fragmentation + full agent loop regression test pass
ames_rf_meta.json  ames_rf.pkl	rf_baseline_meta.json
